In [1]:
import os
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torch.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import csv

# --------- Configuration ---------
DATA_DIR     = r"E:\Learning\UNSW\Term2\9444\group_project\data\split_with_713"
OUTPUT_DIR   = r"E:\Learning\UNSW\Term2\9444\group_project\outputs\plot\cnn9_base"
NUM_CLASSES  = 39
BATCH_SIZE   = 64
NUM_WORKERS  = 8
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_EPOCHS   = 30
WEIGHT_DECAY = 5e-4#1e-4
LR           = 5e-4#1e-3
PATIENCE     = 5

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Enable cuDNN autotuner for fixed-size inputs
torch.backends.cudnn.benchmark = True

# --------- Data Transforms ---------
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
])

# --------- Dataloaders ---------
def get_dataloaders(data_dir, batch_size, num_workers):
    train_ds = datasets.ImageFolder(os.path.join(data_dir, "train"), transform=train_transform)
    val_ds   = datasets.ImageFolder(os.path.join(data_dir, "val"),   transform=val_transform)
    test_ds  = datasets.ImageFolder(os.path.join(data_dir, "test"),  transform=val_transform)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    return train_loader, val_loader, test_loader

# --------- CNN-9 Adjusted Model Definition ---------
class CNN9(nn.Module):
    def __init__(self, num_classes):
        super(CNN9, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 256), nn.ReLU(), nn.Dropout(0.7),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = CNN9(NUM_CLASSES).to(DEVICE)

# --------- Optimizer & Scheduler ---------
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
criterion = nn.CrossEntropyLoss()
scaler = GradScaler('cuda')

# --------- Evaluation Function ---------
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE)
            outputs = model(imgs)
            preds = outputs.argmax(dim=1).cpu().tolist()
            all_preds.extend(preds)
            all_labels.extend(labels.tolist())
    return accuracy_score(all_labels, all_preds)

# --------- Training Loop ---------
def train():
    train_loader, val_loader, test_loader = get_dataloaders(DATA_DIR, BATCH_SIZE, NUM_WORKERS)
    best_val_acc = 0.0
    epochs_no_improve = 0

    train_losses, val_accuracies, epochs_list = [], [], []

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        running_loss = 0.0

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            with autocast('cuda'):
                outputs = model(imgs)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * imgs.size(0)

        scheduler.step()
        train_loss = running_loss / len(train_loader.dataset)
        val_acc = evaluate(model, val_loader)

        train_losses.append(train_loss)
        val_accuracies.append(val_acc)
        epochs_list.append(epoch)

        print(f"Epoch {epoch}/{NUM_EPOCHS} - Train Loss: {train_loss:.4f} - Val Acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "best_cnn9_leaf.pth"))
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print(f"Early stopping at epoch {epoch}.")
                break

    test_acc = evaluate(model, test_loader)
    print(f"Best Val Acc: {best_val_acc:.4f} - Test Acc: {test_acc:.4f}")
    
    # --------- Save metrics CSV ---------
    metrics_path = os.path.join(OUTPUT_DIR, 'metrics.csv')
    with open(metrics_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['epoch', 'train_loss', 'val_acc'])
        for e, l, a in zip(epochs_list, train_losses, val_accuracies):
            writer.writerow([e, l, a])

    # --------- Plot and save figures ---------
    # Training Loss
    fig1 = plt.figure()
    plt.plot(epochs_list, train_losses, marker='o')
    plt.title('Training Loss vs. Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(True)
    fig1.savefig(os.path.join(OUTPUT_DIR, 'training_loss.png'), bbox_inches='tight')
    plt.close(fig1)

    # Validation Accuracy
    fig2 = plt.figure()
    plt.plot(epochs_list, val_accuracies, marker='o')
    plt.title('Validation Accuracy vs. Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.grid(True)
    fig2.savefig(os.path.join(OUTPUT_DIR, 'validation_accuracy.png'), bbox_inches='tight')
    plt.close(fig2)

if __name__ == "__main__":
    train()

Epoch 1/30 - Train Loss: 3.0928 - Val Acc: 0.3714
Epoch 2/30 - Train Loss: 2.5748 - Val Acc: 0.5492
Epoch 3/30 - Train Loss: 2.2041 - Val Acc: 0.6251
Epoch 4/30 - Train Loss: 1.9174 - Val Acc: 0.7079
Epoch 5/30 - Train Loss: 1.6672 - Val Acc: 0.7621
Epoch 6/30 - Train Loss: 1.4972 - Val Acc: 0.8006
Epoch 7/30 - Train Loss: 1.3698 - Val Acc: 0.8200
Epoch 8/30 - Train Loss: 1.2447 - Val Acc: 0.8289
Epoch 9/30 - Train Loss: 1.1596 - Val Acc: 0.8554
Epoch 10/30 - Train Loss: 1.0790 - Val Acc: 0.8730
Epoch 11/30 - Train Loss: 1.0111 - Val Acc: 0.9009
Epoch 12/30 - Train Loss: 0.9509 - Val Acc: 0.8605
Epoch 13/30 - Train Loss: 0.8959 - Val Acc: 0.8867
Epoch 14/30 - Train Loss: 0.8469 - Val Acc: 0.8877
Epoch 15/30 - Train Loss: 0.8037 - Val Acc: 0.8792
Epoch 16/30 - Train Loss: 0.7812 - Val Acc: 0.9114
Epoch 17/30 - Train Loss: 0.7363 - Val Acc: 0.9151
Epoch 18/30 - Train Loss: 0.7055 - Val Acc: 0.9212
Epoch 19/30 - Train Loss: 0.6678 - Val Acc: 0.9251
Epoch 20/30 - Train Loss: 0.6539 - Val A